In [1]:
import geopandas as gpd
import gcsfs
import google.auth
import pandas as pd

import world_cup_vars as wc_vars

import polars as pl
from great_tables import GT

credentials, _ = google.auth.default()

GCS_FILE_PATH = wc_vars.GCS_FILE_PATH

In [12]:
daily_stops = gpd.read_parquet(
    f"{GCS_FILE_PATH}fct_daily_scheduled_stops_{wc_vars.event_name}.parquet",
    storage_options = {"token": credentials}
)
# don't need dim_stops

In [15]:
daily_stops.dtypes

key                                                               object
service_date                                              datetime64[ns]
feed_key                                                          object
stop_id                                                           object
feed_timezone                                                     object
daily_arrivals                                                     int64
first_stop_arrival_datetime_pacific                       datetime64[us]
last_stop_departure_datetime_pacific                      datetime64[us]
_feed_valid_from                                     datetime64[us, UTC]
n_hours_in_service                                                 int64
arrivals_per_hour_owl                                            float64
arrivals_per_hour_early_am                                       float64
arrivals_per_hour_am_peak                                        float64
arrivals_per_hour_midday                           

In [21]:
stops_gdf = daily_stops[[
    "feed_key", "stop_id", 
    "stop_name", "route_type_array", "geometry"]
    ]

In [45]:
# how to filter a list column
#daily_stops[(daily_stops.route_type_3 > 0) & 
#    (daily_stops.route_type_array==[2])]

In [39]:
bus_gdf.route_type_array.value_counts()

route_type_array
[3]       910426
[2]            4
[2, 3]         1
[2, 3]         1
[3, 2]         1
           ...  
[3, 2]         1
[2, 3]         1
[3, 2]         1
[2, 3]         1
[3, 2]         1
Name: count, Length: 3275, dtype: int64

In [47]:
rail_gdf = daily_stops[
    daily_stops[["route_type_0", "route_type_1", "route_type_2"]].sum(axis=1) > 0
][["feed_key", "stop_id", "stop_name", "geometry"]]

In [49]:
bus_gdf2 = filter_to_stops_near_poi(
    bus_gdf, stadium_gdf, METERS_PER_MI * 3
)

rail_gdf2 = filter_to_stops_near_poi(
    rail_gdf,
    stadium_gdf,
    METERS_PER_MI * 10,
)

stops_near_stadium = pd.concat([bus_gdf2, rail_gdf2], axis=0, ignore_index=True)

stops_near_stadium.to_parquet(
    f"{GCS_FILE_PATH}stops_near_poi.parquet", filesystem=gcsfs.GCSFileSystem()
)

In [ ]:
import C1_service_by_route as C1

route_gdf = pd.read_parquet(
    f"{GCS_FILE_PATH}fct_daily_schedule_rt_route_direction_summary_world_cup.parquet", 
    filesystem = gcsfs.GCSFileSystem(),
    columns = ["service_date", "schedule_name", "feed_key", "route_id", "route_id_cleaned",
               "route_name", "direction_id", "route_type",
               "shape_id", "shape_array_key", 
               "n_trips", "n_shapes", "num_stop_times", "avg_stops_served"
              ],
    filters = [[
        ("schedule_name", "in", wc_vars.socal_names + wc_vars.bay_area_names),
    ]]
).merge(
    routes_near_stadium,
    on = ["schedule_name", "route_name", "direction_id", "shape_array_key"],
    how = "inner"
).pipe(C1.merge_routes_with_shape_geom)